In [1]:
# Exploratory Data Analysis (EDA)


In [2]:
import pandas as pd

In [49]:
companies = pd.read_csv("../data/raw/linkedin-job-postings/companies/companies.csv")
benefits = pd.read_csv("../data/raw/linkedin-job-postings/jobs/benefits.csv")
salaries = pd.read_csv("../data/raw/linkedin-job-postings/jobs/salaries.csv")
postings = pd.read_csv("../data/raw/linkedin-job-postings/postings.csv")
company_industries = pd.read_csv("../data/raw/linkedin-job-postings/companies/company_industries.csv")
job_skills = pd.read_csv("../data/raw/linkedin-job-postings/jobs/job_skills.csv")
skills = pd.read_csv("../data/raw/linkedin-job-postings/mappings/skills.csv")

## Postings geographic distribution

In [6]:
postings['location'].value_counts().head(20)

location
United States                      8125
New York, NY                       2756
Chicago, IL                        1834
Houston, TX                        1762
Dallas, TX                         1383
Atlanta, GA                        1363
Boston, MA                         1176
Austin, TX                         1083
Charlotte, NC                      1075
Phoenix, AZ                        1059
Washington, DC                      985
Los Angeles, CA                     972
San Francisco, CA                   884
New York City Metropolitan Area     837
Seattle, WA                         818
San Diego, CA                       790
Denver, CO                          787
Philadelphia, PA                    711
Tampa, FL                           659
Miami, FL                           643
Name: count, dtype: int64

## Most in-demand roles in postings

In [9]:
postings['title'].value_counts().head(20)

title
Sales Manager                      673
Customer Service Representative    373
Project Manager                    354
Administrative Assistant           254
Senior Accountant                  238
Executive Assistant                228
Salesperson                        211
Registered Nurse                   210
Receptionist                       204
Staff Accountant                   200
Account Executive                  195
Retail Sales Associate             190
Sales Associate                    189
Software Engineer                  181
Controller                         175
Account Manager                    171
Store Manager                      166
Senior Software Engineer           162
Assistant Manager                  161
ASSISTANT STORE MANAGER            161
Name: count, dtype: int64

## Most applied jobs in postings

In [23]:
postings[['title', 'applies']].sort_values('applies', ascending=False).head(20)


,title,applies
296,Care Coordinator,967.0
45881,Administrative Specialist - Remote,729.0
43154,UI/UX Designer,625.0
48742,Senior Data Engineer,566.0
2256,Sr Professional Recruiter (Remote) - RPO Consu...,530.0
73254,Associate Software Engineer,508.0
38680,SQL Developer,493.0
65991,Social Media Manager,472.0
73710,Data Engineer,470.0
53525,Full Stack Engineer,465.0


## Postings work type
2 attributes: formatted_work_type and work_type

In [10]:
postings['work_type'].value_counts()

work_type
FULL_TIME     98814
CONTRACT      12117
PART_TIME      9696
TEMPORARY      1190
INTERNSHIP      983
VOLUNTEER       562
OTHER           487
Name: count, dtype: int64

In [11]:
postings['formatted_work_type'].value_counts()

formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64

In [12]:
pd.crosstab(postings['work_type'], postings['formatted_work_type'])


formatted_work_type,Contract,Full-time,Internship,Other,Part-time,Temporary,Volunteer
work_type,,,,,,,
CONTRACT,12117,0,0,0,0,0,0
FULL_TIME,0,98814,0,0,0,0,0
INTERNSHIP,0,0,983,0,0,0,0
OTHER,0,0,0,487,0,0,0
PART_TIME,0,0,0,0,9696,0,0
TEMPORARY,0,0,0,0,0,1190,0
VOLUNTEER,0,0,0,0,0,0,562


In [13]:
postings[['work_type', 'formatted_work_type']].isna().mean()

work_type              0.0
formatted_work_type    0.0
dtype: float64

In [14]:
postings['formatted_work_type'].value_counts().head(20)

formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64

In [18]:
postings['remote_allowed'].value_counts(dropna=False)

remote_allowed
NaN    108603
1.0     15246
Name: count, dtype: int64

In [19]:
(postings['remote_allowed'] == 1).sum()

np.int64(15246)

In [20]:
(postings['remote_allowed'] == 0).sum()

np.int64(0)

## Company size vs quantity of postings

postings -> company_id, company_name

companies -> company_id, name, company_size

In [34]:
postings_per_company = (
    postings
    .groupby('company_id')
    .size()
    .reset_index(name='num_postings')
)
company_postings_size = postings_per_company.merge(
    companies[['name','company_id', 'company_size']],
    on='company_id',
    how='left'
)


In [35]:
company_postings_size.head()

,company_id,num_postings,name,company_size
0,1009.0,33,IBM,7.0
1,1016.0,53,GE HealthCare,7.0
2,1025.0,14,Hewlett Packard Enterprise,7.0
3,1028.0,93,Oracle,7.0
4,1033.0,20,Accenture,7.0


In [38]:
company_postings_size.isna().mean()

company_id      0.000000
num_postings    0.000000
name            0.000082
company_size    0.113386
dtype: float64

In [40]:
company_postings_size.sort_values(
    'num_postings', ascending=False
).head(20)


,company_id,num_postings,name,company_size
19827,53345529.0,1108,Liberty Healthcare and Rehabilitation Services,5.0
6310,167757.0,1003,The Job Network,2.0
21102,73013724.0,604,J. Galt,3.0
195,2152.0,529,TEKsystems,7.0
550,4128.0,527,"Lowe's Companies, Inc.",7.0
384,3175.0,517,Ingersoll Rand,7.0
67,1419.0,496,Capital One,7.0
5851,163139.0,476,Cogent Communications,5.0
1529,11056.0,418,Insight Global,5.0
987,6849.0,415,Dice,5.0


## Most frequent benefits offered in jobs


In [44]:
benefits['type'].value_counts().head(20)

type
401(k)                     24231
Medical insurance           9873
Vision insurance            9309
Disability insurance        7930
Dental insurance            6868
Tuition assistance          2614
Commuter benefits           2226
Paid maternity leave        1808
Paid paternity leave        1540
Pension plan                 906
Student loan assistance      365
Child care support           273
Name: count, dtype: int64

In [45]:
benefits[benefits['inferred'] == 0]['type'].value_counts().head(20)


type
Medical insurance          5264
Dental insurance           5082
Vision insurance           4989
401(k)                     4687
Disability insurance       1850
Paid maternity leave       1512
Paid paternity leave       1287
Tuition assistance         1125
Commuter benefits           675
Pension plan                480
Student loan assistance     295
Child care support          273
Name: count, dtype: int64

## Most frequent industries
postings -> company_id, company_name
company_industries -> company_id, industry

In [47]:
postings_with_industries = postings.merge(
    company_industries[['company_id', 'industry']],
    on='company_id',
    how='left'
)

In [48]:
postings_with_industries['industry'].value_counts().head(20)

industry
Staffing and Recruiting               18886
Hospitals and Health Care             15753
IT Services and IT Consulting         11573
Retail                                 9642
Software Development                   5729
Financial Services                     5516
Construction                           1969
Hospitality                            1913
Non-profit Organizations               1904
Real Estate                            1868
Insurance                              1753
Truck Transportation                   1709
Industrial Machinery Manufacturing     1641
Business Consulting and Services       1590
Higher Education                       1578
Government Administration              1407
Telecommunications                     1372
Food and Beverage Services             1348
Advertising Services                   1334
Motor Vehicle Manufacturing            1331
Name: count, dtype: int64

## Most in-demand skills in postings
postings -> job_id
job_skills -> job_id, skill_abr
skills -> skill_abr, skill_name


In [50]:
job_skills_named = job_skills.merge(
    skills[['skill_abr', 'skill_name']],
    on='skill_abr',
    how='left'
)

In [52]:
job_skills_named['skill_name'].value_counts().head(20)

skill_name
Information Technology    26137
Sales                     22475
Management                20861
Manufacturing             18185
Health Care Provider      17369
Business Development      14290
Engineering               13009
Other                     12608
Finance                    8540
Marketing                  5525
Accounting/Auditing        5461
Administrative             4860
Customer Service           4292
Project Management         3997
Analyst                    3858
Research                   2986
Human Resources            2647
Legal                      2371
Consulting                 2338
Education                  2290
Name: count, dtype: int64